In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib as plt

import folium

import pydeck as pdk

from shapely import Point

import osmnx as ox

#Import trig stuff from math
from math import sin, cos, sqrt, atan2,radians

pd.set_option('display.max_columns', None) ### This line makes all the columns display, rather than ellipses shorten
pd.options.mode.chained_assignment = None  # default='warn'

In [2]:
parcel_and_tax_frame = gpd.read_file("C:/Users/aaron/Fun Mapping Work/Data/Value_Capture_Study/PARCEL_ADDRESS_PUB_AREA_3069_4362620268901785570")

parcel_and_tax_frame['ACCNT_NUM'] = pd.to_numeric(parcel_and_tax_frame['ACCNT_NUM'], errors= 'coerce')

C:\Users\aaron\AppData\Roaming\Python\Python314\site-packages\pyogrio\raw.py:200: RuntimeWarning: C:/Users/aaron/Fun Mapping Work/Data/Value_Capture_Study/PARCEL_ADDRESS_PUB_AREA_3069_4362620268901785570/Parcels_for_King_County_with_Address,_Property_and_Ownership_Information.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [3]:
old_data = pd.read_csv("C:/Users/aaron/Fun Mapping Work/Data/Value_Capture_Study/Maybe Older data/Tax Data/EXTR_TaxAcctReceivable_V.csv")

data_2018 = old_data[old_data['BillYr'] == 2018]
data_2027 = old_data[old_data['BillYr'] == 2027]


In [4]:
combined_frame_2018 = pd.merge(left = parcel_and_tax_frame[['ADDR_FULL', 'ACCNT_NUM', 'geometry', 'LAT', 'LON', 'LOTSQFT', 'KCTP_CITY', 'PROP_NAME']], right = data_2018, left_on='ACCNT_NUM', right_on='AcctNbr')
combined_frame_2027 = pd.merge(left = parcel_and_tax_frame[['ADDR_FULL', 'ACCNT_NUM', 'geometry', 'LAT', 'LON', 'LOTSQFT', 'KCTP_CITY', 'PROP_NAME']], right = data_2027, left_on='ACCNT_NUM', right_on='AcctNbr')

In [5]:
originalCRS = combined_frame_2018.crs

In [6]:
seattle_city_place = "Seattle, Washington, USA"

### includes some of the water area, but overall correct
seattle_boundary = ox.geocode_to_gdf(seattle_city_place)

seattle_boundary = seattle_boundary.to_crs(originalCRS)

In [7]:

combined_frame_2027 = combined_frame_2027.to_crs(originalCRS)
combined_frame_2018 = combined_frame_2018.to_crs(originalCRS)


In [8]:
Seattle_sized_2027_frame = gpd.clip(combined_frame_2027, seattle_boundary, keep_geom_type=False, sort=False)
Seattle_sized_2018_frame = gpd.clip(combined_frame_2018, seattle_boundary, keep_geom_type=False, sort=False)

In [9]:
Seattle_sized_2027_frame['Land_Value_Per_SQFT'] = combined_frame_2027['ApprLandVal'] / combined_frame_2027['LOTSQFT']
Seattle_sized_2018_frame['Land_Value_Per_SQFT'] = combined_frame_2018['ApprLandVal'] / combined_frame_2018['LOTSQFT']

In [10]:
northgateCoords = (47.702990, -122.328066)## LR station
rooseveltCoords = (47.676066, -122.316071)## LR station
UDistrictCoords = (47.660539, -122.314164) ## Brooklyn and 44th (UDistrictStation)
caphillCoords = (47.619004, -122.320646)## LR station
mtBakerCoords = (47.576404, -122.297400)## LR station
columbiaCityCoords = (47.559471, -122.292434) ## LR station
othelloCoords = (47.537818, -122.281678)## LR station
rainierBeachCoords = (47.522383, -122.279250)## LR station

## some non LR station points
ballardCoords = (47.668652, -122.376219) ## 15th and MKT
ravennaCoords = (47.675744, -122.290313) ##35th and 65th
CrownHillCoords = (47.690584, -122.376921) ## 15th and 85th
MagnoliaCoords = (47.646849, -122.400949) ## 34th and Barret


northgatePoint = Point(northgateCoords[1], northgateCoords[0])
rooseveltPoint = Point(rooseveltCoords[1], rooseveltCoords[0])
UDistrictPoint = Point(UDistrictCoords[1], UDistrictCoords[0])
capHillPoint = Point(caphillCoords[1], caphillCoords[0])
mtBakerPoint = Point(mtBakerCoords[1], mtBakerCoords[0])
columbiaCityPoint = Point(columbiaCityCoords[1], columbiaCityCoords[0])
othelloPoint = Point(othelloCoords[1], othelloCoords[0])
rainierBeachPoint = Point(rainierBeachCoords[1], rainierBeachCoords[0])


ballardPoint = Point(ballardCoords[1], ballardCoords[0])
ravennaPoint = Point(ravennaCoords[1], ravennaCoords[0])
crownHillPoint = Point(CrownHillCoords[1], CrownHillCoords[0])
magnoliaPoint = Point(MagnoliaCoords[1], MagnoliaCoords[0])


In [29]:
## create a new frame around a point

def circleFrame(radius, frame):

    gdf_to_return = frame.copy()


    ## project the roosevelt point into a projected CRS
    #roosevelt_point_series = gpd.GeoSeries([rooseveltPoint], crs="EPSG:4326")
    
    point_series = gpd.GeoSeries([rooseveltPoint, northgatePoint, UDistrictPoint, capHillPoint, mtBakerPoint, columbiaCityPoint, 
                                  othelloPoint, rainierBeachPoint, ballardPoint, ravennaPoint, crownHillPoint, magnoliaPoint], crs="EPSG:4326")
    roosevelt_point_projected = point_series.to_crs(originalCRS).iloc[0]
    northgate_point_projected = point_series.to_crs(originalCRS).iloc[1]
    uDistrict_point_projected = point_series.to_crs(originalCRS).iloc[2]
    capHill_point_projected = point_series.to_crs(originalCRS).iloc[3]
    mtBaker_point_projected = point_series.to_crs(originalCRS).iloc[4]
    columbiaCity_point_projected = point_series.to_crs(originalCRS).iloc[5]
    othello_point_projected = point_series.to_crs(originalCRS).iloc[6]
    rainierBeach_point_projected = point_series.to_crs(originalCRS).iloc[7]
    ballard_point_projected = point_series.to_crs(originalCRS).iloc[8]
    ravenna_point_projected = point_series.to_crs(originalCRS).iloc[9]
    crownHill_point_projected = point_series.to_crs(originalCRS).iloc[10]
    magnolia_point_projected = point_series.to_crs(originalCRS).iloc[11]
    
    roosevelt_distance_series = gdf_to_return.distance(roosevelt_point_projected)
    northgate_distance_series = gdf_to_return.distance(northgate_point_projected)
    uDistrict_distance_series = gdf_to_return.distance(uDistrict_point_projected)
    capHill_distance_series = gdf_to_return.distance(capHill_point_projected)
    mtBaker_distance_series = gdf_to_return.distance(mtBaker_point_projected)
    columbiaCity_distance_series = gdf_to_return.distance(columbiaCity_point_projected)
    othello_distance_series = gdf_to_return.distance(othello_point_projected)
    rainierBeach_distance_series = gdf_to_return.distance(rainierBeach_point_projected)
    ballard_distance_series = gdf_to_return.distance(ballard_point_projected)
    ravenna_distance_series = gdf_to_return.distance(ravenna_point_projected)
    crownHill_distance_series = gdf_to_return.distance(crownHill_point_projected)
    magnolia_distance_series = gdf_to_return.distance(magnolia_point_projected)
    gdf_to_return['distance_from_roosevelt'] = roosevelt_distance_series
    gdf_to_return['distance_from_northgate'] = northgate_distance_series
    gdf_to_return['distance_from_uDistrict'] = uDistrict_distance_series
    gdf_to_return['distance_from_capHill'] = capHill_distance_series
    gdf_to_return['distance_from_mtBaker'] = mtBaker_distance_series
    gdf_to_return['distance_from_columbiaCity'] = columbiaCity_distance_series
    gdf_to_return['distance_from_othello'] = othello_distance_series
    gdf_to_return['distance_from_rainierBeach'] = rainierBeach_distance_series
    gdf_to_return['distance_from_ballard'] = ballard_distance_series
    gdf_to_return['distance_from_ravenna'] = ravenna_distance_series
    gdf_to_return['distance_from_crownHill'] = crownHill_distance_series
    gdf_to_return['distance_from_magnolia'] = magnolia_distance_series

    gdf_to_return_non_LR = gdf_to_return[
        (gdf_to_return['distance_from_ballard'] < radius) |
        (gdf_to_return['distance_from_ravenna'] < radius) |
        (gdf_to_return['distance_from_crownHill'] < radius) |
        (gdf_to_return['distance_from_magnolia'] < radius)
        ]

    #print (roosevelt_distance_series)
    gdf_to_return_north = gdf_to_return[
        (gdf_to_return['distance_from_roosevelt'] < radius) |
        (gdf_to_return['distance_from_northgate'] < radius) |
        (gdf_to_return['distance_from_uDistrict'] < radius) |
        (gdf_to_return['distance_from_capHill'] < radius)
        ]

    gdf_to_return_south = gdf_to_return[
        (gdf_to_return['distance_from_mtBaker'] < radius) |
        (gdf_to_return['distance_from_columbiaCity'] < radius) |
        (gdf_to_return['distance_from_othello'] < radius) |
        (gdf_to_return['distance_from_rainierBeach'] < radius)
        ]
    
    gdf_to_return = gdf_to_return[
        (gdf_to_return['distance_from_roosevelt'] < radius) |
        (gdf_to_return['distance_from_northgate'] < radius) |
        (gdf_to_return['distance_from_uDistrict'] < radius) |
        (gdf_to_return['distance_from_capHill'] < radius) |
        (gdf_to_return['distance_from_mtBaker'] < radius) |
        (gdf_to_return['distance_from_columbiaCity'] < radius) |
        (gdf_to_return['distance_from_othello'] < radius) |
        (gdf_to_return['distance_from_rainierBeach'] < radius)
    ]
    #gdf_to_return = gdf_to_return[(roosevelt_distance_series < radius) | (northgate_distance_series < radius) | (uDistrict_distance_series < radius)]
    #gdf_to_return = gdf_to_return[(gdf_to_return['distance_from_roosevelt'] < radius)]
  
    return gdf_to_return_north

In [30]:
#roosevelt_and_northgateframe_2027.head(3)

In [31]:
###lose a bunch of columns before plotting to save some Memory

seattle_frame_2027_trimmed = Seattle_sized_2027_frame.drop([
    "KCTP_CITY",
    "ReceivableType",
    "OmitYr",
    "LevyCode",
    "ApprImpsVal",
    "NewDollar",
    "LandVal",
    "ImpsVal",
    "TaxValReason",
    "TaxStat",
    "AcctStat"], axis=1)

In [32]:
polycentric_frame_2027 = circleFrame(2100, seattle_frame_2027_trimmed)

In [33]:
tt = {'LandValueChange'}

In [34]:
#polycentric_frame_2027.explore(column = 'Land_Value_Per_SQFT', tooltip = tt, vmax = 400)

In [35]:
#break

In [36]:
### append the 2018 land value column onto the 2027 frame
only_accnt_and_lndval_2018 = combined_frame_2018[['ACCNT_NUM', 'ApprLandVal']]

only_accnt_and_lndval_2018 = only_accnt_and_lndval_2018.rename(columns={'ACCNT_NUM': 'ACCNT_NUM_2018', 'ApprLandVal': 'ApprLandVal_2018'})

only_accnt_and_lndval_2018.shape                                  

(610249, 2)

In [37]:
joint_year_polycentric = pd.merge(left = polycentric_frame_2027, right=only_accnt_and_lndval_2018, left_on='ACCNT_NUM', right_on='ACCNT_NUM_2018', how = 'inner')

In [38]:
joint_year_polycentric['LandValueChange'] = joint_year_polycentric['ApprLandVal'] / joint_year_polycentric['ApprLandVal_2018']

In [39]:
###get 
joint_year_polycentric_trimmed = joint_year_polycentric[np.isfinite(joint_year_polycentric['LandValueChange'])]

In [40]:
joint_year_polycentric_trimmed.explore(column = 'LandValueChange', vmax = 4)#, tooltip = tt)

In [41]:
#roosevelt_frame_2027 = circleFrame(rooseveltCoords, 1, combined_frame_2027)
#roosevelt_frame_2018 = circleFrame(rooseveltCoords, 1, combined_frame_2018)
#roosevelt_frame_joint = circleFrame(rooseveltCoords, 1, county_frame_joint)

In [24]:
#roosevelt_frame_2027.shape
#roosevelt_frame_joint.shape

In [25]:
#roosevelt_frame_joint.head()

In [26]:
#roosevelt_frame_joint['LandValueChange'] = roosevelt_frame_joint['ApprLandVal'] / roosevelt_frame_joint['ApprLandVal_2018']

In [27]:
tt = {'ApprLandVal', 'ApprLandVal_2018', 'LandValueChange', 'Land_Value_Per_SQFT'}

In [28]:
roosevelt_frame_joint.explore(column = 'LandValueChange', tooltip = tt, vmax = 4)

NameError: name 'roosevelt_frame_joint' is not defined